Cell 1 — Pip installs

In [ ]:
!pip install -q \
    transformers==4.41.2 \
    accelerate==0.29.3 \
    bitsandbytes>=0.43.3 \
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    pymupdf \
    faiss-cpu \
    gradio \
    sentence-transformers==2.7.0

!pip install -q bitsandbytes --upgrade
!pip install -q fpdf2

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 22.3 MB/s eta 0:00:00


Cell 2 — blooms_tagger.py

In [ ]:
%%writefile blooms_tagger.py
from dataclasses import dataclass
from typing import List
from collections import defaultdict

LEVEL_ORDER = ["Remember", "Understand", "Apply", "Analyze", "Evaluate", "Create"]

LEVEL_EMOJIS = {
    "Remember":   "🟦",
    "Understand": "🟩",
    "Apply":      "🟨",
    "Analyze":    "🟧",
    "Evaluate":   "🟥",
    "Create":     "🟪",
}

BLOOM_KEYWORDS = {
    "Remember":   ["what is", "define", "list", "name", "identify", "recall", "state", "who", "when", "where"],
    "Understand": ["explain", "describe", "summarize", "compare", "interpret", "classify", "paraphrase", "discuss"],
    "Apply":      ["use", "solve", "calculate", "demonstrate", "show how", "apply", "compute", "construct"],
    "Analyze":    ["why", "examine", "differentiate", "break down", "analyze", "distinguish", "compare and contrast"],
    "Evaluate":   ["justify", "assess", "evaluate", "recommend", "judge", "critique", "defend", "argue"],
    "Create":     ["design", "propose", "formulate", "construct", "develop", "create", "invent", "compose"],
}

@dataclass
class TaggedQuestion:
    question: str
    level: str
    confidence: str
    matched_keywords: List[str]
    emoji: str

def tag_question(question: str) -> TaggedQuestion:
    q_lower = question.lower()
    scores = defaultdict(int)
    matched = defaultdict(list)

    for level, keywords in BLOOM_KEYWORDS.items():
        for kw in keywords:
            if kw in q_lower:
                scores[level] += 1
                matched[level].append(kw)

    if scores:
        best_level = max(scores, key=scores.get)
        confidence = "high" if scores[best_level] > 1 else "low"
        return TaggedQuestion(
            question=question,
            level=best_level,
            confidence=confidence,
            matched_keywords=matched[best_level],
            emoji=LEVEL_EMOJIS[best_level],
        )
    else:
        return TaggedQuestion(
            question=question,
            level="Remember",
            confidence="low",
            matched_keywords=[],
            emoji=LEVEL_EMOJIS["Remember"],
        )

def bloom_summary(tagged_questions: List[TaggedQuestion]) -> dict:
    summary = defaultdict(int)
    for tq in tagged_questions:
        summary[tq.level] += 1
    return dict(summary)

Writing blooms_tagger.py


Cell 3 — Load model and define all functions

In [ ]:
import re
import torch
import fitz
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from blooms_tagger import tag_question, bloom_summary, LEVEL_EMOJIS, LEVEL_ORDER

# ── Model ──────────────────────────────────────────────────────────────────────
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
)
print("⏳ Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("⏳ Loading model (this takes ~1-2 min)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
print("✅ Model ready")

# ── RAG components ─────────────────────────────────────────────────────────────
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
_vectorstore_cache = {}

BLOOM_VERBS = {
    "Remember":   "use verbs like 'What is', 'Define', 'List', 'Name', 'Identify'",
    "Understand": "use verbs like 'Explain', 'Describe', 'Summarize', 'Compare', 'Interpret'",
    "Apply":      "use verbs like 'Use', 'Solve', 'Calculate', 'Demonstrate', 'Show how'",
    "Analyze":    "use verbs like 'Why', 'Examine', 'Differentiate', 'Break down', 'Analyze'",
    "Evaluate":   "use verbs like 'Justify', 'Assess', 'Evaluate', 'Recommend', 'Judge'",
    "Create":     "use verbs like 'Design', 'Propose', 'Formulate', 'Construct', 'Develop'",
}

# ── Helper functions ───────────────────────────────────────────────────────────
def load_pdf(path):
    doc = fitz.open(path)
    return "\n".join(page.get_text() for page in doc)

def build_vectorstore(text):
    chunks = splitter.split_text(text)
    return FAISS.from_texts(chunks, embeddings)

def generate_quiz(topic, vectorstore, num_questions=5, selected_levels=None):
    docs = vectorstore.similarity_search(topic, k=3)
    context = "\n\n".join(d.page_content for d in docs)

    if selected_levels and len(selected_levels) > 0:
        level_instruction = f"Generate ALL questions strictly at these Bloom's Taxonomy level(s) only: {', '.join(selected_levels)}.\n"
        for lvl in selected_levels:
            level_instruction += f"- {lvl}: {BLOOM_VERBS[lvl]}\n"
    else:
        level_instruction = "Distribute questions across ALL Bloom's Taxonomy levels:\n"
        for lvl, verbs in BLOOM_VERBS.items():
            level_instruction += f"- {lvl}: {verbs}\n"

    prompt = f"""<|system|>
You are an expert educator and exam setter. Generate exactly {num_questions} multiple-choice questions about the given topic using the context provided.

IMPORTANT: {level_instruction}

Rules for creating questions:
- Each question must have exactly 4 options (A, B, C, D)
- The correct answer should NOT always be B or C, vary it across A, B, C and D
- Wrong options (distractors) must be plausible and believable, not obviously wrong
- Distractors should be related to the topic but subtly incorrect
- STRICTLY follow the Bloom's level, do not generate Remember or Knowledge level questions
- Never add any notes or explanations after the Answer letter
- Answer must be strictly one letter only: A, B, C, or D
- The correct answer must be distributed as: Q1=A, Q2=D, Q3=B, Q4=C, Q5=A
- Never use options like "None of the above" or "All of the above"
- Each distractor should sound like it could be correct to someone who hasnt studied well
- Vary the position of the correct answer across all questions

Format strictly as:
Q1: <question>
Options: A) <opt> B) <opt> C) <opt> D) <opt>
Answer: <letter>

Do not add any extra commentary.<|end|>
<|user|>
Topic: {topic}
Context:
{context}<|end|>
<|assistant|>"""

    from transformers import TextIteratorStreamer
    from threading import Thread

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=True
    )

    generation_kwargs = dict(
        **inputs,
        streamer=streamer,
        max_new_tokens=800,
        temperature=0.7,
        do_sample=True,
        use_cache=True,
    )

    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    generated_text = ""
    for new_text in streamer:
        generated_text += new_text
        yield generated_text

    result = generated_text.strip()

    for tag in ["<|end|>", "<|user|>", "<|assistant|>", "Do not add any extra commentary."]:
        result = result.split(tag)[0].strip()
    yield result

def parse_questions(quiz_text):
    pattern = re.compile(r"(?:Q\d+[:.)])\s*(.+?)(?=\nOptions:|\nAnswer:|\nQ\d+|$)", re.IGNORECASE | re.DOTALL)
    return [m.group(1).strip() for m in pattern.finditer(quiz_text)]

def tag_and_annotate(quiz_text, selected_levels=None):
    questions = parse_questions(quiz_text)

    if selected_levels and len(selected_levels) == 1:
        forced_level = selected_levels[0]
        forced_emoji = LEVEL_EMOJIS[forced_level]
        annotated = quiz_text
        for i in range(1, len(questions) + 1):
            label = f"  [{forced_emoji} {forced_level}]"
            annotated = re.sub(
                rf"(Q{i}[:.)].*?)(\n)",
                lambda m: m.group(1) + label + m.group(2),
                annotated, count=1
            )
        summary_md = f"### Bloom's Taxonomy Distribution\n| Level | Count |\n|---|---|\n| {forced_emoji} {forced_level} | {len(questions)} |"
        return annotated, summary_md

    tagged = [tag_question(q) for q in questions]
    annotated = quiz_text
    for i, tq in enumerate(tagged, 1):
        label = f"  [{tq.emoji} {tq.level}]"
        annotated = re.sub(rf"(Q{i}[:.)].*?)(\n)", lambda m: m.group(1) + label + m.group(2), annotated, count=1)

    summary = bloom_summary(tagged)
    rows = [f"| {LEVEL_EMOJIS[l]} {l} | {summary.get(l, 0)} |" for l in LEVEL_ORDER if summary.get(l, 0) > 0]
    summary_md = "### Bloom's Taxonomy Distribution\n| Level | Count |\n|---|---|\n" + "\n".join(rows)
    return annotated, summary_md

def process(pdf_file, topic, num_questions, selected_levels):
    if pdf_file is None:
        yield "⚠️ Please upload a PDF.", "", ""
        return
    path = pdf_file if isinstance(pdf_file, str) else pdf_file.name
    if path not in _vectorstore_cache:
        print(f"📄 Processing PDF...")
        text = load_pdf(path)
        _vectorstore_cache[path] = build_vectorstore(text)
        print("✅ PDF indexed")
    vs = _vectorstore_cache[path]

    for partial_quiz in generate_quiz(topic, vs, num_questions=int(num_questions), selected_levels=selected_levels):
        annotated, summary = tag_and_annotate(partial_quiz, selected_levels=selected_levels)
        yield partial_quiz, annotated, summary

def download_quiz(raw_quiz):
    if not raw_quiz:
        return None

    import tempfile
    import os

    tmp_dir = tempfile.gettempdir()
    file_path = os.path.join(tmp_dir, "Generated_Quiz.txt")

    with open(file_path, "w", encoding="utf-8") as f:
        f.write("GENERATED QUIZ\n")
        f.write("=" * 50 + "\n\n")
        f.write(raw_quiz)
        f.write("\n\n" + "=" * 50)

    return file_path

print("✅ All functions ready")

⏳ Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ All functions ready


Cell 4 — Launch Gradio UI

In [ ]:
import gradio as gr
import re

def parse_quiz_for_attempt(raw_quiz):
    questions = []
    clean_quiz = raw_quiz
    for tag in ["<|end|>", "<|user|>", "<|assistant|>"]:
        clean_quiz = clean_quiz.split(tag)[0].strip()

    pattern = re.compile(r'Q\d+:', re.IGNORECASE)
    parts = pattern.split(clean_quiz)
    parts = [p.strip() for p in parts if p.strip()]

    for part in parts:
        lines = [l.strip() for l in part.split('\n') if l.strip()]
        if not lines:
            continue
        question_text = lines[0]
        options = []
        answer = ""
        for line in lines[1:]:
            if line.lower().startswith("options:"):
                continue
            elif re.match(r'^[A-D]\)', line):
                options.append(line)
            elif line.lower().startswith("answer:"):
                answer = line.split(":")[-1].strip()[0].upper()
        if question_text and len(options) == 4 and answer:
            questions.append({
                "question": question_text,
                "options": options,
                "answer": answer
            })
    return questions


def build_attempt_ui(raw_quiz):
    questions = parse_quiz_for_attempt(raw_quiz)
    if not questions:
        return [gr.update(visible=False)] * 5 + [
            gr.update(value="⚠️ No questions found. Please generate a quiz first.", visible=True),
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=False),
        ]
    updates = []
    for i in range(5):
        if i < len(questions):
            q = questions[i]
            choices = [opt[3:].strip() for opt in q["options"]]
            updates.append(gr.update(
                label=f"Q{i+1}: {q['question']}",
                choices=choices,
                value=None,
                visible=True
            ))
        else:
            updates.append(gr.update(visible=False))
    return updates + [
        gr.update(value="", visible=False),
        gr.update(visible=True),
        gr.update(visible=False),
        gr.update(visible=False),
    ]


def submit_answers(raw_quiz, a1, a2, a3, a4, a5):
    questions = parse_quiz_for_attempt(raw_quiz)
    if not questions:
        return gr.update(value="⚠️ No questions to evaluate.", visible=True), gr.update(visible=False)

    user_answers = [a1, a2, a3, a4, a5]
    score = 0
    total = len(questions)
    result_text = "📊 QUIZ RESULTS\n" + "="*40 + "\n\n"

    for i, q in enumerate(questions):
        if i >= 5:
            break
        user_ans = user_answers[i]
        correct_letter = q["answer"]
        correct_text = ""
        for opt in q["options"]:
            if opt.startswith(correct_letter + ")"):
                correct_text = opt[3:].strip()
                break
        if user_ans is None:
            status = "⚪ Skipped"
        elif user_ans == correct_text:
            status = "✅ Correct"
            score += 1
        else:
            status = "❌ Wrong"
        result_text += f"Q{i+1}: {q['question']}\n"
        result_text += f"Your Answer  : {user_ans if user_ans else 'Not answered'}\n"
        result_text += f"Correct Answer: {correct_letter}) {correct_text}\n"
        result_text += f"Status: {status}\n\n"

    percentage = (score / total) * 100
    if percentage == 100:
        grade = "🏆 Excellent!"
    elif percentage >= 80:
        grade = "🌟 Very Good!"
    elif percentage >= 60:
        grade = "👍 Good"
    elif percentage >= 40:
        grade = "📚 Keep Studying"
    else:
        grade = "💪 Need More Practice"

    result_text += "="*40 + "\n"
    result_text += f"SCORE: {score}/{total} ({percentage:.0f}%)\n"
    result_text += f"GRADE: {grade}\n"
    result_text += "="*40

    return gr.update(value=result_text, visible=True), gr.update(visible=True)


def reset_attempt():
    return [gr.update(value=None, visible=False)] * 5 + [
        gr.update(value="", visible=False),
        gr.update(visible=False),
        gr.update(visible=False),
        gr.update(visible=False),
    ]


with gr.Blocks(title="Bloom's Quiz Generator") as demo:
    gr.Markdown("# 📚 Bloom's Taxonomy Quiz Generator\nUpload a PDF, enter a topic, and get auto-tagged questions.")

    with gr.Row():
        pdf_input = gr.File(label="📄 Upload PDF", file_types=[".pdf"])
        topic_input = gr.Textbox(label="🔍 Topic", placeholder="e.g. taxonomic categories")
        num_q = gr.Slider(minimum=3, maximum=5, value=3, step=1, label="Number of Questions")

    level_selector = gr.CheckboxGroup(
        choices=["Remember", "Understand", "Apply", "Analyze", "Evaluate", "Create"],
        value=[],
        label="🎯 Bloom's Level Filter (leave empty for all levels)",
    )

    generate_btn = gr.Button("🚀 Generate Quiz", variant="primary")

    with gr.Tabs():
        with gr.Tab("📝 Raw Quiz"):
            raw_output = gr.Textbox(label="Generated Quiz", lines=20)
            download_btn = gr.Button("⬇️ Download Quiz as Text File")
            download_file = gr.File(label="📥 Click below to download", interactive=False)

        with gr.Tab("🏷️ Tagged Quiz"):
            tagged_output = gr.Textbox(label="Quiz with Bloom's Tags", lines=20)

        with gr.Tab("📊 Distribution"):
            summary_output = gr.Markdown()

        with gr.Tab("✏️ Attempt Quiz"):
            attempt_btn = gr.Button("📝 Start Quiz Attempt", variant="secondary")
            q1 = gr.Radio(label="Q1", choices=[], visible=False)
            q2 = gr.Radio(label="Q2", choices=[], visible=False)
            q3 = gr.Radio(label="Q3", choices=[], visible=False)
            q4 = gr.Radio(label="Q4", choices=[], visible=False)
            q5 = gr.Radio(label="Q5", choices=[], visible=False)
            msg = gr.Textbox(label="", visible=False, interactive=False)
            submit_btn = gr.Button("✅ Submit Answers", variant="primary", visible=False)
            result_box = gr.Textbox(label="📊 Results", lines=20, visible=False)
            retry_btn = gr.Button("🔄 Try Again", visible=False)

            attempt_btn.click(
                fn=build_attempt_ui,
                inputs=[raw_output],
                outputs=[q1, q2, q3, q4, q5, msg, submit_btn, result_box, retry_btn]
            )

            submit_btn.click(
                fn=submit_answers,
                inputs=[raw_output, q1, q2, q3, q4, q5],
                outputs=[result_box, retry_btn]
            )

            retry_btn.click(
                fn=reset_attempt,
                inputs=[],
                outputs=[q1, q2, q3, q4, q5, msg, submit_btn, result_box, retry_btn]
            )

    generate_btn.click(
        fn=process,
        inputs=[pdf_input, topic_input, num_q, level_selector],
        outputs=[raw_output, tagged_output, summary_output],
    )

    download_btn.click(
        fn=download_quiz,
        inputs=[raw_output],
        outputs=[download_file]
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e0d628516a886bd2e9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr

with gr.Blocks(title="Bloom's Quiz Generator") as demo:
    gr.Markdown("# 📚 Bloom's Taxonomy Quiz Generator\nUpload a PDF, enter a topic, and get auto-tagged questions.")

    with gr.Row():
        pdf_input = gr.File(label="📄 Upload PDF", file_types=[".pdf"])
        topic_input = gr.Textbox(label="🔍 Topic", placeholder="e.g. taxonomic categories")
        num_q = gr.Slider(minimum=3, maximum=10, value=5, step=1, label="Number of Questions")

    level_selector = gr.CheckboxGroup(
        choices=["Remember", "Understand", "Apply", "Analyze", "Evaluate", "Create"],
        value=[],
        label="🎯 Bloom's Level Filter (leave empty for all levels)",
    )

    generate_btn = gr.Button("🚀 Generate Quiz", variant="primary")

    with gr.Tabs():
        with gr.Tab("📝 Raw Quiz"):
            raw_output = gr.Textbox(label="Generated Quiz", lines=20)

            download_btn = gr.Button("⬇️ Download Quiz as Text File")
            download_file = gr.File(label="📥 Click below to download", interactive=False)
        with gr.Tab("🏷️ Tagged Quiz"):
            tagged_output = gr.Textbox(label="Quiz with Bloom's Tags", lines=20)
        with gr.Tab("📊 Distribution"):
            summary_output = gr.Markdown()

    generate_btn.click(
        fn=process,
        inputs=[pdf_input, topic_input, num_q, level_selector],
        outputs=[raw_output, tagged_output, summary_output]
    )

    download_btn.click(
        fn=download_quiz,
        inputs=[raw_output],
        outputs=[download_file]
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://863c2cd5cabbfdd8d5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# Debug cell
test_text = """Q1: Test question?
Options: A) Option1 B) Option2 C) Option3 D) Option4
Answer: A"""

result = download_quiz(test_text)
print("Result:", result)

Result: /tmp/Generated_Quiz.txt
